# DS2002 · Git for Data Science

**Lecture — 2026-08-31 · Fall 2026**  
**Class time:** 45 minutes

---

## Connect Colab to your GitHub repo (do this first)

Everything you submit this semester lands in **your own** repository. Before we touch Git commands, get Colab talking to that repo. Find your situation below.

**You already did Lab 01.** You have `https://github.com/<you>/ds2002-fa26`. You are reconnecting Colab to it, not starting over.

**You never made the repo, or you cannot find it.** Go to <https://github.com/new>, name it `ds2002-fa26`, check **Add a README file**, and create it. That is the whole setup — you connect Colab in the steps below.

**It used to work and now it does not, or you are signed in as the wrong person.** In Colab, open the **Files** panel on the left, pick the **GitHub** tab, and use **Connect to GitHub**. If it shows someone else's account, disconnect first, then reconnect. Sign in as the account that owns *your* repo — not a lab partner's.

### Connect and clone

1. Open this notebook in **Google Colab**.
2. **Files -> GitHub -> Connect to GitHub**, and finish the authorization prompt.
3. Clone your repo into the runtime:

```bash
!git clone https://github.com/<you>/ds2002-fa26.git
%cd ds2002-fa26
```

4. Check that it worked: `!ls -la` and `!git remote -v`.

The clone lives at `/content/ds2002-fa26` and only lasts for this session. When the runtime disconnects, run the clone cell again — nothing is lost, because the real copy is on GitHub.

### If you would rather not clone yet

**File -> Save a copy in GitHub** pushes just this notebook. Choose *your* `ds2002-fa26` repo and a path like `notebooks/01-foundations/`. That is enough for one notebook, but once we are pushing every week the clone is less work.

**On Kaggle instead?** Kaggle has no GitHub connection. Keep to the Lab 01 route — download the notebook and push from your machine, or upload it through the GitHub web interface. Every Git command in this lecture still runs in Kaggle.

The cells further down work in a throwaway sandbox folder so you can break things safely. The connection you just made is for your real repo, all semester.

In [ ]:
import sys, os, subprocess

in_colab = 'google.colab' in sys.modules
print('Environment:', 'Colab' if in_colab else 'Kaggle or other')

# TODO: your repo URL from Lab 01, or the one you just created
MY_REPO = 'https://github.com/TODO/ds2002-fa26'
print('My repo:', MY_REPO)

CLONE_DIR = '/content/ds2002-fa26'   # change if you cloned somewhere else
if os.path.isdir(os.path.join(CLONE_DIR, '.git')):
    subprocess.run('git remote -v && git log --oneline -3',
                   shell=True, cwd=CLONE_DIR)
else:
    print('No clone yet -- run the clone cell above, or use '
          'File -> Save a copy in GitHub for now.')

### When the connection fights back

| Problem | Fix |
|---|---|
| No **Connect to GitHub** option appears | Use Chrome, turn off blockers, or try an incognito window |
| The wrong GitHub account is connected | GitHub tab -> manage the connection -> disconnect -> reconnect |
| `Authentication failed` when pushing | GitHub stopped accepting passwords in 2021. Use a [personal access token](https://github.com/settings/tokens) as the password, or push with **Save a copy in GitHub** |
| Runtime reset and the folder is gone | Run `!git clone ...` again; your commits are safe on GitHub |
| You never created a repo | Make `ds2002-fa26` on GitHub first, then come back |

## Where a change lives

Friday you made commits. Today we go past the happy path, because the commands that matter in a real project are the ones you reach for when something is already wrong.

Every change you make is sitting in exactly one of four places. Almost every confusing Git moment is really "I thought my change was in a different place than it is."

```
working directory  --add-->  staging area  --commit-->  local repo  --push-->  remote
```

| Place | What it holds | How you see it |
|---|---|---|
| Working directory | Edits you have saved to disk | `git status`, `git diff` |
| Staging area | What you have chosen for the next commit | `git status`, `git diff --staged` |
| Local repo | Committed history on your machine | `git log` |
| Remote | The copy on GitHub | GitHub, `git log origin/main` |

`git status` answers "which place is my work in?" Run it more than you think you need to.

### A repo to work in

Same sandbox idea as Friday. We build a small repo so every command below prints real output instead of a description of output.

In [ ]:
import subprocess, tempfile, os

SANDBOX = tempfile.mkdtemp(prefix='ds2002-git-')

def git(cmd, cwd=SANDBOX):
    """Run a command in the sandbox folder and print exactly what git says back."""
    done = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print('$', cmd)
    out = (done.stdout + done.stderr).strip()
    print(out if out else '(no output)')
    print()
    return done

print('sandbox:', SANDBOX)
git('git --version')

In [ ]:
git('git init -b main')
git("git config user.name 'DS2002'")
git("git config user.email 'ds2002@example.com'")

def write(path, text):
    with open(os.path.join(SANDBOX, path), 'w') as f:
        f.write(text)

write('prices.csv', 'item,price\nponcho,6.00\nburger,7.50\n')
git('git add prices.csv')
git("git commit -m 'Add vendor price list'")

write('clean.py', 'def clean(df):\n    return df.drop_duplicates()\n')
git('git add clean.py')
git("git commit -m 'Add first cleaning step'")
git('git log --oneline')

### Reading history

Three questions come up constantly, and each has one command.

**"What changed recently?"**

In [ ]:
git('git log --oneline --stat')

**"What exactly did that commit do?"** `git show` takes any commit hash. `HEAD` means the current commit, and `HEAD~1` means the one before it.

In [ ]:
git('git show HEAD~1')

**"What have I changed since my last commit?"** This is the one to run before every commit. Note the difference between unstaged and staged changes.

In [ ]:
write('prices.csv', 'item,price\nponcho,6.00\nburger,7.50\nnachos,5.25\n')

git('git diff')              # not staged yet
git('git add prices.csv')
git('git diff')              # empty now -- it moved to staging
git('git diff --staged')     # here it is

### Undoing things

Four situations, four different commands. Using the wrong one is how people lose work.

| Situation | Command | Destructive? |
|---|---|---|
| Staged something by mistake | `git restore --staged <file>` | No |
| Want to throw away an uncommitted edit | `git restore <file>` | **Yes** — the edit is gone |
| A commit was wrong, and it is already pushed | `git revert <hash>` | No — adds an undo commit |
| "Just make it look like before" | `git reset --hard` | **Yes** — avoid it |

Unstaging first, since it is the common one:

In [ ]:
git('git restore --staged prices.csv')
git('git status --short')     # back to unstaged: the edit still exists
git('git add prices.csv')
git("git commit -m 'Add nachos to price list'")

`git revert` is the one to know for shared work. It does not erase history — it writes a new commit that undoes an old one, so anyone who already pulled stays in sync.

In [ ]:
git('git log --oneline')
git('git revert --no-edit HEAD')
git('git log --oneline')
print(open(os.path.join(SANDBOX, 'prices.csv')).read())

Four commits now: nachos was added, then reverted, and both facts are in the history. That is the behavior you want on a team — nothing silently disappears.

### Two people, one file

This is where project weeks go wrong, so let's cause the problem on purpose. A branch is just a movable label pointing at a commit; working on one lets you change files without disturbing `main`.

In [ ]:
git('git checkout -b cheaper-ponchos')
write('prices.csv', 'item,price\nponcho,4.00\nburger,7.50\n')
git("git commit -am 'Drop poncho price to 4.00'")

git('git checkout main')
write('prices.csv', 'item,price\nponcho,8.00\nburger,7.50\n')
git("git commit -am 'Raise poncho price to 8.00'")

git('git log --oneline --graph --all')

The same line changed on both branches. Git cannot pick a winner, so merging stops and asks you.

In [ ]:
git('git merge cheaper-ponchos')
print('--- prices.csv as Git left it ---')
print(open(os.path.join(SANDBOX, 'prices.csv')).read())

Those `<<<<<<<`, `=======`, `>>>>>>>` markers are Git showing you both versions. Resolving means editing the file until it is what you want, removing the markers, then adding and committing. There is no magic command — it is a human decision.

In [ ]:
write('prices.csv', 'item,price\nponcho,6.00\nburger,7.50\n')  # the decision
git('git add prices.csv')
git("git commit -m 'Resolve price conflict: settle on 6.00'")
git('git log --oneline --graph')

### Notebooks make this worse

A `.ipynb` is JSON with your code, your outputs, and an execution counter per cell. When two people edit one notebook, the conflict lands inside that JSON and the markers break the file so it will not even open.

Rules that keep you out of that hole on the midterm and capstone:

1. **One owner at a time** for any given notebook. Say so out loud in Discord.
2. **Pull before you start working**, not after you finish.
3. **Restart and Run All, then commit.** Committed outputs should match committed code.
4. Split project work into **separate files** — cleaning, API, analysis — instead of three people in one notebook.
5. If a notebook does conflict, the fastest honest fix is usually to keep one side whole, re-apply the other person's cells by hand, and re-run.

### Commit messages that earn their keep

You are writing to whoever opens `git log` in three weeks, which is usually you.

| Weak | Better |
|---|---|
| `update` | `Lab 03: add HAVING query for Q4` |
| `fixed stuff` | `Drop duplicate NC State export rows` |
| `asdf` | `Normalize vendor_id formats before the join` |
| `final version` | `Use rate per 1k attendance instead of raw units` |

One idea per commit. If your message needs the word "and" twice, it should probably have been two commits.

### Practice — read the state, then fix it

The cell below leaves the sandbox in a specific state. Run it, then use `git status` and `git diff` to answer, in the markdown cell after it: which of the four places is each change sitting in?

In [ ]:
write('README.md', '# Game day pipeline\n')
write('clean.py', 'def clean(df):\n    return df.drop_duplicates().dropna()\n')
git('git add README.md')

# TODO: run the two commands that show you what is staged and what is not

**Your answer:** `README.md` is in _____ and `clean.py` is in _____.

### Practice — commit them separately

Both files changed for different reasons, so they should be two commits with two messages. Do that, then show the log.

In [ ]:
# TODO: commit README.md with its own message
# TODO: then stage and commit clean.py with its own message
# TODO: git log --oneline

### For the rest of the semester

Wednesday we deal with the other half of reproducibility: notebooks that only run because of the order you happened to click cells in. Friday's lab is a debugging drill plus your first full submission through the real loop.